
# AuRA v0.7.6 — Assessment State & Dynamic Question Queue

**AuRA (Autism Risk Analytics)** is an AI chatbot-based autism spectrum
disorder (ASD) pre-screening system integrating standardized assessment
tools and machine-learning-based risk classification.

This version focuses on managing assessment state, maintaining a dynamic
question queue, and accumulating behavioral feature results.

The system also includes an informational ASD knowledge base, which
provides educational responses independently of the assessment pipeline.

To avoid confusion, AuRA does not rely on a **Large Language Model (LLM)** to process user responses or generate its primary screening results. Instead, it uses **rule-based Natural Language Processing (NLP) techniques, built-in Python libraries, and machine learning algorithms implemented through libraries** such as Scikit-learn and XGBoost. This approach allows the system to process behavioral descriptions, extract relevant indicators, and classify preliminary ASD risk levels without requiring the development or training of a large language model.

**Important:** AuRA supports pre-screening and educational awareness.
It does not diagnose ASD or replace professional clinical evaluation.


## 1. Imports and System Configuration

This section imports the Python libraries required for text processing,
fuzzy matching, JSON handling, and file management.

It also defines the system version, debugging configuration, behavioral
feature names, supported languages, and other shared constants.

**Purpose:** Establish the configuration and dependencies used by the
remaining modules.

In [1]:
# 1. Imports

import string
import re
import difflib
import json
from pathlib import Path

import textwrap
# This wouldn't be need anyway in NiceGUI, 
# this is merely for debugging's readability.

print(Path.cwd())

VERSION = "0.7.6.3"
DEBUG_MODE = True

DATABASE_PATH = Path("..") / "databases" / VERSION

DATABASE_FILE = (
    DATABASE_PATH /
    f"behavioral_indicators_{VERSION}.json"
)
medical_library_PATH = Path("..") / "databases" / VERSION

MEDICAL_DATABASE_FILE = (
    medical_library_PATH /
    f"medical_library_{VERSION}.json"
)

with open(DATABASE_FILE, "r", encoding="utf-8") as file:
    behavioral_indicators = json.load(file)
with open(MEDICAL_DATABASE_FILE, "r", encoding="utf-8") as file:
    medical_library = json.load(file)

c:\PROGRAMMING STUFF\Thesis - AuRA\Python Notebooks


In [2]:
# Imports and initialization of taglish terms

import re
import unicodedata

TAGLISH_NORMALIZATION = {
    # Filipino abbreviations
    "hnd": "hindi",
    "d": "di",
    "di": "hindi",
    "nde": "hindi",
    "nd": "hindi",
    "wag": "huwag",
    "wagng": "huwag",
    "lng": "lang",
    "lang": "lang",
    "minsan": "minsan",
    "palagi": "palagi",
    "sya": "siya",
    "nya": "niya",
    "syang": "siyang",
    "pla": "pala",
    "nmn": "naman",
    "dn": "din",
    "rin": "rin",
    "oo": "oo",
    "opo": "opo",
    "hndi": "hindi",
    "tlga": "talaga",
    "kse": "kasi",
    "kasi": "kasi",
    "ung": "yung",
    "yng": "yung",
    "bsta": "basta",
    "kng": "kung",

    # common taglish frequency words
    "always": "always",
    "usually": "usually",
    "sometimes": "sometimes",
    "occasionally": "occasionally",
    "rarely": "rarely",
    "never": "never",

    # Common Taglish negation and contractions
    "dont": "dont",
    "doesnt": "doesnt",
    "cant": "cant",

    # Common Taglish verb forms for Q1 - response_to_name
    # ADD MORE FOR OTHER FEATURES
    "nagre-respond": "respond",
    "nagrerespond": "respond",
    "nagrespond": "respond",
    "nagre-respond": "respond",

    # Common Taglish verb forms for Q2 - eye_contact
}


## 2. Behavioral Indicator Database Loading

This section loads the behavioral-indicator definitions from the
corresponding JSON file.

The database contains the terms and phrases used by the NLP module to
recognize the behavioral feature currently being assessed.

**Input:** Behavioral-indicator JSON file.

**Output:** A dictionary of behavioral indicators and their language-
specific matching terms.

**Note:** This database is separate from the ASD Informational
Knowledge Base.

In [3]:
## 2. Initialization of Files and Variables
# Q1-Q9 : Expected Behavior Features

# Frequency values:
# 0 = Never / Wala
# 1 = Sometimes / Minsan
# 2 = Always / Palagi

EXPECTED_BEHAVIOR_FEATURES = [
    "response_to_name",
    "eye_contact",
    "joint_attention",
    "gesture_use",
    "social_interaction",
    "sharing_interest",
    "play_behavior",
    "language_communication",
    "following_instructions"
]

# Q10-Q13 : Concerning / "Red-Flag" Features

# Frequency values:

CONCERNING_BEHAVIOR_FEATURES = [
    "repetitive_behaviors_routines",
    "sensory_responses",
    "emotional_response_to_change",
    "fixated_restricted_interest"
]

# Frequency Values

EXPECTED_BEHAVIOR_VALUES = {
    "tierA": 0, # Always / Palagi
    "tierB": 1, # Sometimes / Minsan
    "tierC": 2  # Never / Hindi
}

CONCERNING_BEHAVIOR_VALUES = {
    "tierA": 0, # No /Wala
    "tierB": 1, # Sometimes / Minsan
    "tierC": 2  # Yes / Meron
}

# Frequency Expressions for Expected Behavior Questions

EXPECTED_BEHAVIOR_FREQEX = {
    "en": {
        "tierA": ["always", "usually", "every time"],
        "tierB": ["sometimes", "occasionally", "rarely"],
        "tierC": ["never", "not at all"]
    },
    "tl": {
        "tierA": ["palagi", "lagi", "laging", "tuwing"],
        "tierB": ["minsan", "bihira", "paminsan"],
        "tierC": ["hindi", "wala", "never"]
    }
}


# Frequency Expressions for Concerning or "Red-flag" Questions

CONCERNING_BEHAVIOR_FREQEX = {
    "en": {
        "tierA": ["no", "not at all", "never"],
        "tierB": ["sometimes", "occasionally", "rarely"],
        "tierC": ["yes", "always", "usually", "every time"]
    },
    "tl": {
        "tierA": ["hindi", "wala"],
        "tierB": ["minsan", "bihira", "paminsan"],
        "tierC": ["meron", "tuwing", "may ganito", "may ganitong"]
    }
}

# Tier Labels

TIER_LABELS = {
    "expected_behavior": {
        "en": {
            "tierA": "always",
            "tierB": "sometimes",
            "tierC": "never"
        },
        "tl": {
            "tierA": "palagi",
            "tierB": "minsan",
            "tierC": "hindi"
        }
    },
    "concerning_behavior": {
        "en": {
            "tierA": "no",
            "tierB": "sometimes",
            "tierC": "yes"
        },
        "tl": {
            "tierA": "wala",
            "tierB": "minsan",
            "tierC": "meron"
        }
    }
}


NEGATION_WORDS = {
    "no",
    "not",
    "dont",
    "doesnt",
    "isnt",
    "cant",
    "cannot",
    "never",
    "without",
    "neither",
    "nor",
    "nothing",
    "nowhere",
    "none",
    "nobody",

    "hindi",
    "wala",
    "di",
    "ayaw"
}

QUESTIONS = {
    "en": [
        {
            "id": "Q1",
            "indicator": "response_to_name",
            "question": "Does your child respond when you call their name?"
        }
    ],

    "tl": [
        {
            "id": "Q1",
            "indicator": "response_to_name",
            "question": "Tumutugon ba ang iyong anak kapag tinatawag mo ang kanyang pangalan?"
        }
    ]
}

def validate_behavioral_indicators(indicators):
    expected_features = (
        EXPECTED_BEHAVIOR_FEATURES
        + CONCERNING_BEHAVIOR_FEATURES
    )

    errors = []

    if set(indicators) != set(expected_features):
        errors.append("Database keys do not match the 13 expected features.")

    seen_indices = []

    for feature in expected_features:
        data = indicators.get(feature)

        if not isinstance(data, dict):
            errors.append(f"{feature}: missing or invalid entry.")
            continue

        expected_category = (
            "expected_behavior"
            if feature in EXPECTED_BEHAVIOR_FEATURES
            else "concerning_behavior"
        )

        if data.get("category") != expected_category:
            errors.append(f"{feature}: incorrect category.")

        index = data.get("feature_index")

        if not isinstance(index, int) or isinstance(index, bool):
            errors.append(f"{feature}: invalid feature_index.")
        else:
            seen_indices.append(index)
            if index != expected_features.index(feature):
                errors.append(f"{feature}: incorrect feature_index.")

        for language in ("en", "tl"):
            language_data = data.get(language)

            if not isinstance(language_data, dict):
                errors.append(f"{feature}: missing {language} dictionary.")
                continue

            for key in (
                "feature_terms",
                "positive_terms",
                "concern_terms",
                "negation_terms"
            ):
                if not isinstance(language_data.get(key), list):
                    errors.append(
                        f"{feature}.{language}.{key} must be a list."
                    )

    if len(seen_indices) != len(set(seen_indices)):
        errors.append("Duplicate feature_index values found.")

    if errors:
        raise ValueError("\n".join(errors))

    print("PASS: Behavioral-indicator database is structurally valid.")


## 3. Text Preprocessing Functions

This section prepares user input for NLP processing.

Operations may include:
- Converting text to lowercase.
- Removing punctuation.
- Normalizing whitespace.
- Tokenizing the normalized text.

**Input:** The parent's free-form response.

**Output:** Normalized text and tokens for downstream processing.

Text preprocessing does not independently determine the meaning or
clinical significance of a response.


In [4]:
## 2. TEXT PREPROCESSING FUNCTIONS

def normalize_text(text):

    text = text.lower()

    tokens = text.split()
    tokens = [
        TAGLISH_NORMALIZATION.get(token, token)
        for token in tokens
        ]

    text = " ".join(tokens)
    
    # remove apostrophes and punctuation
    text = text.replace("'", "")

    text = re.sub(
        f"[{re.escape(string.punctuation)}]",
        "",
        text
    )

    # collapse excessive whitespace
    text = " ".join(text.split())

    return text


def tokenize_text(text):
    return normalize_text(text).split()


## 4. ASD Informational Knowledge Base

This module provides educational information about Autism Spectrum
Disorder (ASD) using the project's curated informational knowledge base.

The source material covers:
- ASD definitions and characteristics.
- Signs, symptoms, and developmental differences.
- Early identification and screening.
- M-CHAT-R/F, Q-CHAT, and CSBS-DP.
- Diagnosis and associated risk factors.
- Interventions and professional referrals.
- Frequently asked questions.

The module searches for relevant topics and returns the corresponding
informational response and its source reference.

**Input:** A general informational question from the user.

**Output:** A matched knowledge-base entry, response, and source when
available.

**Boundary:** This module provides education, not diagnosis or risk
classification. It must remain separate from the assessment's feature
results and question index.

In [5]:
## MEDICAL LIBRARY / ASD KNOWLEDGE BASE


## 5. Behavioral-Indicator Detection

This section identifies the behavioral indicator relevant to the
current assessment question.

It uses the behavioral-indicator database to match phrases in the
parent's response. Depending on the implementation, matching may use
exact phrases and fuzzy matching.

**Input:** Normalized text, language, and the current target indicator.

**Output:** Indicator-match information, including the matched phrase
and match type when available.

If no relevant indicator is detected, the system should request
clarification rather than infer an answer.


In [6]:
## Detect Behavioral Indicators
# (STRICTLY DETECT IF THE RESPONSE 
# IS MATCHED WITH THE CURRENT QUESTION)

def detect_behavioral_indicator(tokens, language, target_indicator):

    # indicator_data = behavioral_indicators[target_indicator][language]
    indicator_data = get_indicator_data(target_indicator, language)

    # More informative behavior terms take priority over context phrases.
    term_groups = [
        ("concern", indicator_data["concern_terms"]),
        ("positive", indicator_data["positive_terms"]),
        ("feature", indicator_data["feature_terms"]),
    ]

    matches = []

    for term_type, phrases in term_groups:
        for phrase in phrases:
            phrase_tokens = tokenize_text(phrase)
            size = len(phrase_tokens)

            if not size:
                continue

            for i in range(len(tokens) - size + 1):
                if tokens[i:i + size] == phrase_tokens:
                    matches.append({
                        "indicator": target_indicator,
                        "matched_phrase": phrase,
                        "match_type": "exact",
                        "term_type": term_type,
                        "start": i,
                        "end": i + size,
                    })

    if not matches:
        return None

    # Priority: explicit concern > positive > feature.
    priority = {
        "concern": 0,
        "positive": 1,
        "feature": 2,
    }

    matches.sort(
        key=lambda match: (
            priority[match["term_type"]],
            -(match["end"] - match["start"]),
        )
    )

    return matches[0]


## 6. Polarity and Context Resolution

This section interprets a detected behavioral indicator in context.

The current design distinguishes between:
- Neutral or expected behavior.
- Concerning behavior.

Negation terms and the type of matched phrase help determine the
context. This is a rule-based interpretation and should be tested
against positive, negative, and ambiguous statements.

**Input:** Tokens and indicator-match information.

**Output:** The resolved behavioral context.

Context resolution is distinct from frequency detection and numerical
feature encoding.


In [7]:
def resolve_polarity(tokens, indicator_match, language):

    if indicator_match is None:
        return None

    indicator = indicator_match["indicator"]
    term_type = indicator_match["term_type"]
    start = indicator_match["start"]

    indicator_data = behavioral_indicators[indicator][language]

    negation_terms = [
        tokenize_text(term)
        for term in indicator_data.get("negation_terms", [])
    ]

    # Check for negation immediately before the matched behavior.
    has_negation = False

    for phrase_tokens in negation_terms:
        size = len(phrase_tokens)

        if size == 0 or start < size:
            continue

        if tokens[start - size:start] == phrase_tokens:
            has_negation = True
            break

    if term_type == "positive":
        return "concern" if has_negation else "neutral"

    if term_type == "concern":
        return "concern"

    # A general feature phrase does not provide enough
    # evidence to determine the behavior's polarity.
    return None

def normalize_frequency(context, frequency, target_indicator):

    if frequency is None:
        return None

    if target_indicator in EXPECTED_BEHAVIOR_FEATURES:

        if context == "neutral":
            if frequency in ("tierA", "tierB"):
                return frequency

        elif context == "concern":
            if frequency in ("tierA", "tierB", "tierC"):
                return frequency

    elif target_indicator in CONCERNING_BEHAVIOR_FEATURES:

        if frequency in ("tierA", "tierB", "tierC"):
            return frequency

    return None

In [8]:
# Helper to combine English and Filipino expressions for Taglish support

def get_language_expressions(expressions_by_language, language):
    selected = expressions_by_language.get(language, {})

    if language != "tl":
        return selected

    english = expressions_by_language.get("en", {})
    merged = {}

    for tier in set(selected) | set(english):
        merged[tier] = list(dict.fromkeys(
            selected.get(tier, []) + english.get(tier, [])
        ))

    return merged

def get_indicator_data(target_indicator, language):
    indicator = behavioral_indicators[target_indicator]

    selected = indicator.get(language, {})

    # English-only mode stays unchanged.
    if language != "tl":
        return selected

    english = indicator.get("en", {})
    merged = {}

    for key in (
        "feature_terms",
        "positive_terms",
        "concern_terms",
        "negation_terms",
    ):
        merged[key] = list(dict.fromkeys(
            selected.get(key, []) + english.get(key, [])
        ))

    return merged


## 7. Frequency Detection

This section searches the parent's response for recognized frequency
expressions in the selected language.

Examples include "never," "sometimes," and "always," along with their
configured Tagalog equivalents.

If the response does not specify frequency, the assessment may request
a structured follow-up answer.

**Input:** Tokens, language, and target indicator.

**Output:** A detected frequency tier or no detected frequency.

A missing frequency must not be treated as though the NLP successfully
identified one.


In [9]:
## Frequency Detection

# Detects frequency expressions.

# This updated version of the functions improve phrase matching but it does not solve negation.

def detect_frequency(tokens, language, target_indicator):
    if target_indicator in EXPECTED_BEHAVIOR_FEATURES:
        expressions = get_language_expressions(EXPECTED_BEHAVIOR_FREQEX, language)

    elif target_indicator in CONCERNING_BEHAVIOR_FEATURES:
        expressions = get_language_expressions(CONCERNING_BEHAVIOR_FREQEX, language)

    else:
        raise ValueError(
            f"Unknown behavioral feature: {target_indicator}"
        )

    matches = []

    NEGATION_ONLY = {
        "hindi", "di", "wala",
        "not", "does not", "doesnt", "never"
    }

    # Match longer phrases before shorter phrases
    for tier, phrases in expressions.items():
        for phrase in phrases:
            phrase_tokens = phrase.split()

            # Skip entries that are only negation terms.
            if phrase.strip() in NEGATION_ONLY:
                continue

            size = len(phrase_tokens)

            for i in range(len(tokens) - size + 1):
                if tokens[i:i + size] == phrase_tokens:
                    matches.append((size, tier, phrase))

    if not matches:
        return None

    longest_match = max(match[0] for match in matches)

    longest_matches = [
        match for match in matches
        if match[0] == longest_match
    ]

    # Collect unique tiers, not a generator
    matched_tiers = {match[1] for match in longest_matches}

    if len(matched_tiers) != 1:
        return None

    return next(iter(matched_tiers))

## Frequency Follow-Up

# if the response does not have frequency terms like always, sometimes, rarely, never, etc.

def ask_frequency(target_indicator, context):

    print()

    print("How often does this kind of behavior happen?")

    # expected behavior features
    if (target_indicator in EXPECTED_BEHAVIOR_FEATURES):

        if context == "neutral":

            print("[1] Always / Palagi")
            print("[2] Sometimes / Minsan")

            while True:

                choice = input("> ").strip()

                if choice == "1":
                    return "tierA" # registers as 0

                if choice == "2":
                    return "tierB" # registers as 1

                print("Please choose 1, 2.")

        elif context == "concern":
            print("[1] Always / Palagi")
            print("[2] Sometimes / Minsan")

            while True:
            
                choice = input("> ").strip()

                if choice == "1":
                    return "tierC" # registers as 2

                if choice == "2":
                    return "tierB" # registers as 1


    # concerning behavior features
    elif target_indicator in CONCERNING_BEHAVIOR_FEATURES:
        print("[1] No / Wala")
        print("[2] Sometimes / Minsan")
        print("[3] Yes / Meron")

        while True:
            choice = input("> ").strip()

            if choice == "1":
                return "tierA"
            
            if choice == "2":
                return "tierB"
            
            if choice == "3":
                return "tierC"

            print("Please choose 1, 2, or 3.")

    else:
        raise ValueError(
            f"Unexpected combination of target_indicator and context: {target_indicator}, {context}"
        )


## 8. Explicit Absence Detection

This section handles responses that explicitly indicate the absence
or non-occurrence of a behavioral indicator.

It supports the distinction between:
- The absence of an expected behavior.
- The absence of a concerning or red-flag behavior.

The interpretation depends on the feature category and the assessment
context.

**Input:** The processed response and relevant behavioral feature.

**Output:** An absence-related result for subsequent interpretation.

**Example:**

Expected behavior (Q1, responds_to_name): “My child never responds to his name.”

Expected behavior (Q2, eye_contact): “My child does not make eye contact with me.”

This statement indicates the presence of the expected behavior
associated with `response_to_name`. The system should interpret the
statement in context before assigning a frequency tier or numerical
value.


In [10]:
def is_explicitly_absent(tokens, context, language, target_indicator):
    indicator_data = behavioral_indicators[target_indicator][language]
    
    negation_terms = indicator_data["negation_terms"]

    if context == "concern":
        return True

    for term in negation_terms:
        term_tokens = term.split()

        for i in range(len(tokens) - len(term_tokens) + 1):
            if tokens[i:i + len(term_tokens)] == term_tokens:
                return True

    return False


## 9. Frequency Normalization and Ordinal Feature Encoding

This section converts the interpreted behavioral response into the
system's internal frequency tiers and numerical representation.

The internal tiers are:
- `tierA`
- `tierB`
- `tierC`

The system maps these tiers to numerical values according to the
feature category and its interpretation rules.

For Q1–Q9, the encoding represents concern associated with the absence
of expected behaviors. For Q10–Q13, it represents the occurrence of
red-flag behaviors.

**Output:** A numerical feature value for the corresponding feature.

The encoding rules must remain consistent with the definitions used
when preparing and training the machine-learning models.

## 10. Numerical Feature Vector (NFV) Generation

This section assembles the individual numerical feature values into
a fixed-order vector for machine-learning input.

The current design contains 13 behavioral features, ordered according
to the system's feature definitions and question mapping.

**Input:** The accumulated behavioral feature results.

**Output:** A 13-dimensional Numerical Feature Vector (NFV).

The vector must preserve the expected feature order and numerical
semantics. Missing responses must be handled according to the
assessment's defined missing-value policy.


In [11]:
## Feature Encoding

def encode_frequency(feature, frequency):

    if feature in EXPECTED_BEHAVIOR_FEATURES:
        return EXPECTED_BEHAVIOR_VALUES[frequency]
    
    elif feature in CONCERNING_BEHAVIOR_FEATURES:
        return CONCERNING_BEHAVIOR_VALUES[frequency]
    
    else:
        raise ValueError (f"Unknown behavioral feature: {feature}") 

## NFV Generation

# It should contain 13 ordered features corresponding to Q1-Q13.

def generate_nfv(feature_results):
    expected_features = (
        EXPECTED_BEHAVIOR_FEATURES
        + CONCERNING_BEHAVIOR_FEATURES
    )

    missing_features = [
        feature for feature in expected_features
        if feature not in feature_results
    ]

    invalid_features = [
        feature for feature in expected_features
        if feature in feature_results
        and feature_results[feature] not in ("tierA", "tierB", "tierC")
    ]

    if missing_features or invalid_features:
        raise ValueError(
            f"Missing features: {missing_features}; "
            f"invalid features: {invalid_features}"
        )

    return [
        encode_frequency(feature, feature_results[feature])
        for feature in expected_features
    ]



## 11. Machine-Learning Models

This section contains the machine-learning models used to classify
the numerical feature vector.

The planned models are:
- Logistic Regression.
- Random Forest.
- XGBoost.

Each model must receive the same feature ordering and compatible
numerical input representation used during training.

**Input:** The completed NFV.

**Output:** Individual model predictions for the configured risk
categories.

Model predictions are estimates for pre-screening purposes and are
not clinical diagnoses.


In [ ]:
## MACHINE LEARNING (not yet implemented)

def predict_risk(nfv):

    # TODO
    #
    # LR prediction
    # RF prediction
    # XGBoost prediction
    #
    # Then soft voting.

    return None

In [13]:
## PROCESS RESPONSE

def process_response(text, language, target_indicator):

    # 1. Preprocess
    normalized = normalize_text(text)
    tokens = tokenize_text(normalized)

    # 2. Indicator detection
    indicator_match = detect_behavioral_indicator(
        tokens,
        language,
        target_indicator
    )

    if indicator_match is None:
        return {
            "status": "clarify",
            "normalized": normalized,
            "tokens": tokens,
            "indicator_match": None
        }

    # 3. Context / polarity [variable declared]
    context = resolve_polarity(
        tokens,
        indicator_match,
        language
    )

    # 4. Frequency detection
    # Two categories have different meanings, therefore
    detected_frequency = detect_frequency(
        tokens,
        language,
        target_indicator
    )

    # 5. Normalize semantic frequency
    frequency = normalize_frequency(
        context,
        detected_frequency,
        target_indicator
    )

    ## DEBUG

    print()
    print("DEBUG target_indicator:", target_indicator)
    print("DEBUG indicator_match:", indicator_match)
    print("DEBUG context:", context)
    print("DEBUG detected_frequency:", detected_frequency)
    print()
    
    if frequency is None:
        if context not in ("neutral", "concern"):
            return {
                "status": "needs_followup",
                "context": context,
                "detected_frequency": detected_frequency,
                "indicator_match": indicator_match,
                "context": context
            }
        frequency = ask_frequency(target_indicator, context)

    # 6. Feature encoding
    encoded = encode_frequency(
        target_indicator,
        frequency
    )

    return {
        "status": "success",
        "normalized": normalized,
        "tokens": tokens,
        "indicator_match": indicator_match,
        "context": context,
        "detected_frequency": detected_frequency,
        "frequency": frequency,
        "encoded": encoded
    }

In [14]:
## QUESTION QUEUE

def init_question_queue(indicators):
    queue = []

    for indicator, data in indicators.items():
        feature_index = data.get("feature_index")

        if feature_index is None:
            continue

        queue.append({
            "question_number": feature_index + 1,
            "indicator": indicator,
            "category": data.get("category")
        })

    # sort by actual question number
    queue.sort(key=lambda question: question["question_number"])

    return queue

QUESTION_QUEUE = init_question_queue(behavioral_indicators)

for question in QUESTION_QUEUE:
    print(
        f"Q{question['question_number']}: "
        f"{question['indicator']}"
    )



Q1: response_to_name
Q2: eye_contact
Q3: joint_attention
Q4: gesture_use
Q5: social_interaction
Q6: sharing_interest
Q7: play_behavior
Q8: language_communication
Q9: following_instructions
Q10: repetitive_behaviors_routines
Q11: sensory_responses
Q12: emotional_response_to_change
Q13: fixated_restricted_interest


In [15]:

# AuRA assessment integrity check

EXPECTED_FEATURES = (
    EXPECTED_BEHAVIOR_FEATURES
    + CONCERNING_BEHAVIOR_FEATURES
)

queued_features = [
    question["indicator"]
    for question in QUESTION_QUEUE
]

queued_numbers = [
    question["question_number"]
    for question in QUESTION_QUEUE
]

missing_features = [
    feature for feature in EXPECTED_FEATURES
    if feature not in queued_features
]

unexpected_features = [
    feature for feature in queued_features
    if feature not in EXPECTED_FEATURES
]

print("Expected features:", len(EXPECTED_FEATURES))
print("Queued questions:", len(QUESTION_QUEUE))
print("Question numbers:", queued_numbers)
print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)
print("Duplicate question numbers:",
      sorted({
          number for number in queued_numbers
          if queued_numbers.count(number) > 1
      }))

if (
    len(QUESTION_QUEUE) == 13
    and queued_numbers == list(range(1, 14))
    and not missing_features
    and not unexpected_features
):
    print("PASS: Assessment queue contains Q1-Q13.")
else:
    print("FAIL: Fix the assessment queue before proceeding.")


Expected features: 13
Queued questions: 13
Question numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Missing features: []
Unexpected features: []
Duplicate question numbers: []
PASS: Assessment queue contains Q1-Q13.


In [16]:
# functions for medical library branch
# currently, it is hardcoded to english, change it if you want to support other languages

def normalize_library_query(text):
    return normalize_text(text)

def detect_library_intent(text, language="en"):
    normalized = normalize_library_query(text)

    for topic_id, topic in medical_library.get("topics", {}).items():
        keywords = topic.get("keywords", [])

        if isinstance(keywords, dict):
            keywords = keywords.get(language, [])

        for keyword in keywords:
            normalized_keyword = normalize_library_query(keyword)

            if normalized == normalized_keyword:
                return topic_id

    return None

def retrieve_library_article(article_id):
    return medical_library.get("topics", {}).get(article_id)

def answer_medical_query(text, language="en"):
    article_id = detect_library_intent(text, language)

    if article_id is None:
        return None

    return retrieve_library_article(article_id)

In [17]:

query = "what is autism"
language = "en"

print("1. Library type:", type(medical_library))
print("2. Topic IDs:", list(medical_library.get("topics", {}).keys()))

topic = medical_library["topics"]["autism_definition"]
print("3. Keywords:", repr(topic["keywords"]))
print("4. English keywords:", repr(topic["keywords"].get("en")))

print("5. Normalized query:", repr(normalize_library_query(query)))

print("6. Detected intent:",
      detect_library_intent(query, language))

print("7. Retrieved article:",
      answer_medical_query(query, language))


1. Library type: <class 'dict'>
2. Topic IDs: ['autism_definition', 'autism_screening_vs_diagnosis']
3. Keywords: {'en': ['what is autism', 'what is asd', 'autism spectrum disorder'], 'tl': ['ano ang autism', 'ano po ang autism', 'autism spectrum disorder']}
4. English keywords: ['what is autism', 'what is asd', 'autism spectrum disorder']
5. Normalized query: 'what is autism'
6. Detected intent: autism_definition
7. Retrieved article: {'keywords': {'en': ['what is autism', 'what is asd', 'autism spectrum disorder'], 'tl': ['ano ang autism', 'ano po ang autism', 'autism spectrum disorder']}, 'responses': {'en': 'Autism spectrum disorder (ASD) is a neurological and developmental condition that begins in the developmental period and affects how a person communicates, interacts, learns, and behaves. ', 'tl': 'Ang Autism Spectrum Disorder (ASD) ay isang neurodevelopmental condition na maaaring makaapekto sa komunikasyon, pakikisalamuha, at pag-uugali.'}, 'sources': ['https://www.who.int/ne

In [ ]:
FOLLOWUP_QUESTIONS = {
    "en": {
        "response_to_name":
            "Can you describe what your child usually does when you call their name?",

        "eye_contact":
            "In what situations does your child look toward your face or eyes while interacting with someone?",

        "joint_attention":
            "What does your child do when you point out something interesting to them?",

        "gesture_use":
            "How does your child use gestures, such as pointing, waving, or reaching, to communicate?",

        "social_interaction":
            "Can you describe how your child usually interacts with other children or adults?",

        "sharing_interest":
            "How does your child show or share something interesting with you?",

        "play_behavior":
            "What does your child usually do during play, alone or with other people?",

        "language_communication":
            "How does your child communicate their needs, thoughts, or interests?",

        "following_instructions":
            "Can you describe how your child responds to a simple, familiar instruction?",

        "repetitive_behaviors_routines":
            "Can you describe any repeated movements, actions, or routines you have noticed?",

        "sensory_responses":
            "Are there particular sounds, textures, lights, or other sensations your child reacts to? Please describe.",

        "emotional_response_to_change":
            "What usually happens when an expected activity or routine changes?",

        "fixated_restricted_interest":
            "Are there particular interests or activities your child focuses on extensively? Please describe."
    },

    "tl": {
        "response_to_name":
            "Maaari mo bang ilarawan kung ano ang karaniwang ginagawa ng iyong anak kapag tinatawag mo ang kanyang pangalan?",

        "eye_contact":
            "Sa anong mga sitwasyon tumitingin sa mukha o mga mata ng kausap ang iyong anak?",

        "joint_attention":
            "Ano ang ginagawa ng iyong anak kapag may itinuturo kang kawili-wiling bagay sa kanya?",

        "gesture_use":
            "Paano gumagamit ng mga kilos ang iyong anak, tulad ng pagturo, pagkaway, o pag-abot, upang makipag-usap?",

        "social_interaction":
            "Maaari mo bang ilarawan kung paano karaniwang nakikisalamuha ang iyong anak sa ibang bata o matatanda?",

        "sharing_interest":
            "Paano ipinapakita o ibinabahagi ng iyong anak sa iyo ang isang bagay na kinaiinteresan niya?",

        "play_behavior":
            "Ano ang karaniwang ginagawa ng iyong anak kapag naglalaro nang mag-isa o kasama ang ibang tao?",

        "language_communication":
            "Paano ipinapahayag ng iyong anak ang kanyang mga pangangailangan, iniisip, o interes?",

        "following_instructions":
            "Maaari mo bang ilarawan kung paano tumutugon ang iyong anak sa isang simple at pamilyar na utos?",

        "repetitive_behaviors_routines":
            "Maaari mo bang ilarawan ang mga paulit-ulit na kilos, gawain, o rutin na napansin mo?",

        "sensory_responses":
            "May mga tunog, tekstura, ilaw, o iba pang pandamang partikular na tinutugunan ng iyong anak? Pakilarawan.",

        "emotional_response_to_change":
            "Ano ang karaniwang nangyayari kapag nagbabago ang nakasanayang gawain o iskedyul ng iyong anak?",

        "fixated_restricted_interest":
            "May mga interes o aktibidad bang matagal na pinagkakaabalahan ng iyong anak? Pakilarawan."
    }
}



## Features to add in the system

1. Another Follow-Up Question after answering a frequency of said behavior.
- For example, if the user answers "Always / Palagi" based on their child's behavior, then the follow-up question may also ask specific behaviors like 

2. Results Summary (literally in the name: Autism Risk Analytics)
- It shows Screening Summary (tally), List of Present Behavioral Indicators, Parent-Family Explanation, Suggested Next Steps (and Disclaimer) followed by suggesting available doctors.

3. ML Training
- Provide 500 valid respondents in an Excel file for training

4. Synthetic Data Usage for a test drive in a clinical setting.

In [18]:

# AuRA regression tests — detailed pipeline debugger
# English + Tagalog
#
# Classification is determined ONLY by:
#   detect_behavioral_indicator()
#   resolve_polarity()
#
# Expected labels are used only to evaluate the result.

import pandas as pd
import re


# ---------------------------------------------------------
# 1. DEBUG HELPERS
# ---------------------------------------------------------

def debug_section(title):
    print("\n" + "-" * 76)
    print(title)
    print("-" * 76)


def safe_get(mapping, key, default=None):
    """Safely retrieve a dictionary field."""
    if isinstance(mapping, dict):
        return mapping.get(key, default)
    return default


def show_match_details(match):
    """Display the fields returned by the indicator detector."""
    if match is None:
        print("Indicator match: None")
        return

    if isinstance(match, dict):
        print("Indicator match details:")
        for key, value in match.items():
            print(f"  {key}: {value}")
    else:
        print(f"Indicator match: {match!r}")


def get_feature_terms(feature, language):
    """
    Read the terms for the current feature from the loaded JSON.
    This is diagnostic only; it does not determine classification.
    """
    if "behavioral_indicators" not in globals():
        return {}

    feature_data = behavioral_indicators.get(feature, {})
    language_data = feature_data.get(language, {})

    return language_data if isinstance(language_data, dict) else {}


def show_term_hits(normalized, feature, language):
    """
    Display JSON terms that occur literally in the normalized input.
    These are diagnostic hits, not the detector's final match.
    """
    terms_by_type = get_feature_terms(feature, language)

    print("\nLiteral JSON term hits:")
    found_any = False

    for term_type, terms in terms_by_type.items():
        if not isinstance(terms, list):
            continue

        hits = []
        for term in terms:
            if not isinstance(term, str):
                continue

            normalized_term = normalize_text(term)
            if normalized_term and normalized_term in normalized:
                hits.append(term)

        if hits:
            found_any = True
            print(f"  {term_type}: {hits}")

    if not found_any:
        print("  No literal term hits.")
        print("  Note: fuzzy matching may still find a match.")


def show_negation_context(tokens, feature, language):
    """Show negation tokens and JSON negation terms for inspection."""
    terms_by_type = get_feature_terms(feature, language)
    negation_terms = terms_by_type.get("negation_terms", [])

    print("\nNegation inspection:")
    print(f"  Input tokens: {tokens}")
    print(f"  JSON negation terms: {negation_terms}")

    normalized_negations = {
        normalize_text(term)
        for term in negation_terms
        if isinstance(term, str)
    }

    token_negations = [
        (i, token)
        for i, token in enumerate(tokens)
        if token in normalized_negations
    ]

    print(f"  Exact token-level negation hits: {token_negations}")

    # Also inspect common negation phrases in the normalized input.
    phrase_hits = [
        term for term in negation_terms
        if isinstance(term, str)
        and normalize_text(term)
        and normalize_text(term) in " ".join(tokens)
    ]

    print(f"  Negation phrase hits: {phrase_hits}")



# ---------------------------------------------------------
# 2. TEST INPUTS — NO EXPECTED LABELS
# ---------------------------------------------------------

TEST_CASES = {
    "response_to_name": {
        "en": [
            "My child responds to his name when I call him",
            "My child does not look at me when I call him",
            "my child can respond to his name when i call him.",
            "my son responds to name sometimes",
            "my son always respond to his name when i call him."
        ],
        "tl": [
            "Tumingin ang anak ko kapag tinatawag ko siya",
            "Tumingin ang anak ko sa akin kapag tinawag ko siya.",
            "Hindi tumitingin ang anak ko kapag tinatawag ko siya",
            "Tumutugon siya sa pangalan niya kung tinatawag ko siya.",
            "ang anak ko di nagrerespond sa pangalan niya kapag tinawag ko siya.",
            "ang anak ko di nagrespond kapag tinawag ko sya.",
            "Nagre-respond ang anak ko kapag tinatawag ko siya.",
            "Hindi nagre-respond ang anak ko kapag tinatawag ko siya.",
            "Minsan nagre-respond ang anak ko kapag tinatawag ko siya.",
            "Always siyang tumutugon kapag tinatawag ko siya."
        ],
    },

    "eye_contact": {
        "en": [
            "My child makes eye contact with me.",
            "My child does not make eye contact with me.",
            "My child looks at me when I talk to him.",
            "My child avoids looking into my eyes.",
            "My child sometimes makes eye contact with me.",
            "My child always looks at me when I speak to him."
        ],
        "tl": [
            "Tumingin sa akin ang anak ko kapag kinakausap ko siya.",
            "Hindi tumitingin sa akin ang anak ko kapag kinakausap ko siya.",
            "Nakikipag-eye contact ang anak ko sa akin.",
            "Hindi nakikipag-eye contact ang anak ko sa akin.",
            "Iniiwasan ng anak ko na tumingin sa mga mata ko.",
            "Minsan tumitingin sa akin ang anak ko kapag kinakausap ko siya.",
            "Lagi siyang tumitingin sa akin kapag kinakausap ko siya.",
            "Nag-eye contact ang anak ko sa akin minsan.",
            "Hindi nag-eye contact ang anak ko sa akin.",
            "Sometimes nakikipag-eye contact ang anak ko sa akin.",
            "Always siyang tumitingin sa akin kapag kinakausap ko siya."
        ],
    },
}

# ---------------------------------------------------------
# 3. RUN DETAILED DEBUGGING
# ---------------------------------------------------------

def run_regression_tests(test_cases, verbose=True):
    results = []

    for feature, languages in test_cases.items():
        for language, cases in languages.items():
            for text in cases:

                print("\n" + "=" * 76)
                print(f"TEST: {language} | {feature}")
                print("=" * 76)

                # A. ORIGINAL INPUT
                print(f"Original input: {text!r}")

                # B. NORMALIZATION
                normalized = normalize_text(text)
                print(f"\nNormalized text: {normalized!r}")

                # C. TOKENIZATION
                tokens = tokenize_text(normalized)
                print("\nTokens:")
                for index, token in enumerate(tokens):
                    print(f"  [{index}] {token!r}")

                # D. INDICATOR DETECTION
                print("\n[1] INDICATOR DETECTION")
                try:
                    match = detect_behavioral_indicator(
                        tokens, language, feature
                    )
                    show_match_details(match)
                except Exception as exc:
                    match = None
                    print(f"DETECTOR ERROR: {type(exc).__name__}: {exc}")

                if verbose:
                    show_term_hits(normalized, feature, language)
                    show_negation_context(tokens, feature, language)

                # E. POLARITY RESOLUTION
                print("\n[2] POLARITY RESOLUTION")
                try:
                    context = resolve_polarity(
                        tokens, match, language
                    )
                    print(f"Resolved polarity: {context!r}")
                except Exception as exc:
                    context = None
                    print(f"POLARITY ERROR: {type(exc).__name__}: {exc}")

                # F. FREQUENCY DETECTION
                print("\n[3] FREQUENCY DETECTION")
                try:
                    frequency = detect_frequency(
                        tokens, language, feature
                    )
                    print(f"Detected frequency: {frequency!r}")
                except Exception as exc:
                    frequency = None
                    print(f"FREQUENCY ERROR: {type(exc).__name__}: {exc}")

                # G. PIPELINE STATUS
                if match is None:
                    status = "NO MATCH"
                elif context not in ("neutral", "concern"):
                    status = "CLARIFY"
                elif frequency is None:
                    status = "FOLLOW-UP"
                else:
                    status = "NO FOLLOW-UP"

                print("\n[4] CLASSIFICATION")
                print(f"Feature:   {feature}")
                print(f"Polarity:  {context!r}")
                print(f"Frequency: {frequency!r}")
                print(f"Status:    {status}")

                results.append({
                    "feature": feature,
                    "language": language,
                    "input": text,
                    "normalized": normalized,
                    "tokens": tokens,
                    "match": match,
                    "polarity": context,
                    "frequency": frequency,
                    "status": status,
                })

    df = pd.DataFrame(results)

    print("\n" + "=" * 76)
    print("CLASSIFICATION SUMMARY")
    print("=" * 76)

    display(
        df[
            ["feature", "language", "input",
             "polarity", "frequency", "status"]
        ]
    )

    return df


# ---------------------------------------------------------
# 4. EXECUTE
# ---------------------------------------------------------

test_results_df = run_regression_tests(TEST_CASES)




TEST: en | response_to_name
Original input: 'My child responds to his name when I call him'

Normalized text: 'my child responds to his name when i call him'

Tokens:
  [0] 'my'
  [1] 'child'
  [2] 'responds'
  [3] 'to'
  [4] 'his'
  [5] 'name'
  [6] 'when'
  [7] 'i'
  [8] 'call'
  [9] 'him'

[1] INDICATOR DETECTION
Indicator match details:
  indicator: response_to_name
  matched_phrase: responds
  match_type: exact
  term_type: positive
  start: 2
  end: 3

Literal JSON term hits:
  positive_terms: ['responds', 'respond']

Negation inspection:
  Input tokens: ['my', 'child', 'responds', 'to', 'his', 'name', 'when', 'i', 'call', 'him']
  JSON negation terms: ['does not', 'doesnt', 'never', 'not']
  Exact token-level negation hits: []
  Negation phrase hits: []

[2] POLARITY RESOLUTION
Resolved polarity: 'neutral'

[3] FREQUENCY DETECTION
Detected frequency: None

[4] CLASSIFICATION
Feature:   response_to_name
Polarity:  'neutral'
Frequency: None
Status:    FOLLOW-UP

TEST: en | respon

,feature,language,input,polarity,frequency,status
0,response_to_name,en,My child responds to his name when I call him,neutral,NaN,FOLLOW-UP
1,response_to_name,en,My child does not look at me when I call him,concern,NaN,FOLLOW-UP
2,response_to_name,en,my child can respond to his name when i call him.,neutral,NaN,FOLLOW-UP
3,response_to_name,en,my son responds to name sometimes,neutral,tierB,NO FOLLOW-UP
4,response_to_name,en,my son always respond to his name when i call ...,neutral,tierA,NO FOLLOW-UP
5,response_to_name,tl,Tumingin ang anak ko kapag tinatawag ko siya,neutral,NaN,FOLLOW-UP
6,response_to_name,tl,Tumingin ang anak ko sa akin kapag tinawag ko ...,neutral,NaN,FOLLOW-UP
7,response_to_name,tl,Hindi tumitingin ang anak ko kapag tinatawag k...,concern,NaN,FOLLOW-UP
8,response_to_name,tl,Tumutugon siya sa pangalan niya kung tinatawag...,neutral,NaN,FOLLOW-UP
9,response_to_name,tl,ang anak ko di nagrerespond sa pangalan niya k...,concern,NaN,FOLLOW-UP


In [19]:
# Run the whole chatbot

validate_behavioral_indicators(behavioral_indicators)

def select_language():
    print("\nSelect your language / Pumili ng wika:")
    print("[1] English")
    print("[2] Filipino / Tagalog / Taglish")

    languages = {
        "1": "en",
        "2": "tl"
    }

    while True:
        choice = input("> ").strip()
        print("You:", repr(choice))

        if choice in languages:
            return languages[choice]

        print("Please choose 1 or 2. / Pumili ng 1 or 2.")

def run_assessment(language):
    print("\n=== AuRA Risk Assessment ===")
    print("This is a pre-screening tool, not a medical diagnosis.")
    print("Type 'menu' to return to the main menu.\n")

    assessment = {
        "current_question_index": 0,
        "feature_results": {},
        "completed": False
    }

    TEST_MODE = True
    TEST_FEATURE_LIMIT = 1

    while assessment["current_question_index"] < min(
        len(QUESTION_QUEUE),
        TEST_FEATURE_LIMIT if TEST_MODE else len(QUESTION_QUEUE)
    ):
        question = QUESTION_QUEUE[
            assessment["current_question_index"]
        ]

        question_number = question["question_number"]
        target_indicator = question["indicator"]
    
        print(f"Q{question_number}: {target_indicator}")
        print("Please describe your child's behavior.")
        ## TODO -> put the actual question text here based on the language selected.

        text = input("> ").strip()

        text = input("You: ").strip()
        print("You:", repr(text))

        if normalize_text(text) == "menu":
            print("Returning to main menu...\n")
            return

        result = process_response(text, language, target_indicator)

        if result["status"] == "clarify":
            print("I could not identify the behavior clearly.")
            print("Please describe the behavior more specifically.\n")
            continue

        if result["status"] != "success":
            print("Unable to process this response.\n")
            continue

        print("Detected context:", result["context"])
        print("Detected frequency:", result["detected_frequency"])

        # Ask for clarification if frequency is missing.
        frequency = result["frequency"]
        
        if frequency is None:
            print("Unable to determine frequency. Please try again.\n")
            continue

        assessment["feature_results"][target_indicator] = frequency
        assessment["current_question_index"] += 1

        print("Response recorded.\n")

    assessment["completed"] = True

    print("\nAssessment test completed.")

    if TEST_MODE:
        feature_order = list(dict.fromkeys(
            question["indicator"]
            for question in QUESTION_QUEUE
        ))

        tier_mapping = {
            "tierA": 0,
            "tierB": 1,
            "tierC": 2
        }

        nfv = [
            tier_mapping.get(assessment["feature_results"].get(feature), 0)
            for feature in feature_order
        ]

        print("Tested features:", assessment["feature_results"])
        print("Numerical Feature Vector (NFV):", nfv)
        print("Number of features:", len(nfv))
        print("Note: Untested features are placeholders, not assessed scores.")

    else:
        nfv = generate_nfv(assessment["feature_results"])

        print("Numerical Feature Vector (NFV):", nfv)
        print("Number of features:", len(nfv))

        if len(nfv) != 13:
            print("Error: Expected exactly 13 features.")
            return

def format_response(text, wpl=18):
    words = text.split()
    lines = []

    for i in range(0, len(words), wpl):
        lines.append(" ". join(words[i:i + wpl]))

    return "\n".join(lines)


def run_medical_library(language):

    print("\n=== AuRA Medical Library ===")
    print("Ask an ASD-related informational question.")
    print("Type 'menu' to return to the main menu.\n")

    while True:

        text = input("You: ").strip()
        print("You:", repr(text))

        if normalize_text(text) == "menu":
            print("Returning to the main menu.")
            return

        article = answer_medical_query(text, language)

        if article is None:
            print("AuRA: I couldn't find a matching topic. Try rephrasing your question.")
            continue

        response = article.get("responses", {}).get(language)

        if response is None:
            print("AuRA: No response is available in that language.")
            continue

        print("\nAuRA:")
        print(format_response(response, wpl=10))

        sources = article.get("sources", article.get("source"))

        if sources:
            if isinstance(sources, list):
                print("Sources:")

                for source in sources:
                    print("-", source)
                    
            else:
                print("Source:", source)

        print()

    

# Branch Selection

def select_branch():
    print()
    print("AuRA: Autism Risk Analytics")
    print("How can I help you today?")
    print()
    print("[1] ASD Risk Assessment")
    print("[2] ASD Medical Library")
    print("[3] Exit")

    choices = {
        "1": "assessment",
        "2": "library",
        "3": "exit"
    }

    while True:
        choice = input("> ").strip()

        if choice in choices:
            return choices[choice]

        print("Please choose 1, 2, or 3.")

# Run Chatbot

def run_chatbot():
    language = select_language()

    while True:
        branch = select_branch()

        if branch == "assessment":
            print("\nStarting assessment...\n")
            run_assessment(language)

        elif branch == "library":
            print("\nStarting medical library...\n")
            run_medical_library(language)
        
        elif branch == "exit":
            print("\nThank you for using AuRA. Goodbye!\n")
            break


print("AuRA is starting...")
run_chatbot()


PASS: Behavioral-indicator database is structurally valid.
AuRA is starting...

Select your language / Pumili ng wika:
[1] English
[2] Filipino / Tagalog / Taglish
You: '1'

AuRA: Autism Risk Analytics
How can I help you today?

[1] ASD Risk Assessment
[2] ASD Medical Library
[3] Exit


KeyboardInterrupt: Interrupted by user